# Crossed Book Diagnosis

build_silver fails on 2026-04-18 with 41% crossed book rate.
April 18 has 280 snapshots, 74,818 deltas, **0 trades**.

Since there are no trades, this can't be a double-counting issue.
Hypothesis: early bronze data is missing `conn_id`/`sid` fields,
so the transform can't detect reconnects or filter stale deltas.

This notebook investigates:
1. Do April 18 records have conn_id and sid?
2. How many unique sids exist per channel?
3. Are there reconnects (sid changes) that aren't being detected?
4. What does the book state look like when it crosses?

In [ ]:
import boto3
import gzip
import json
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

s3 = boto3.client("s3")
BUCKET = "prediction-markets-data"
SOURCE = "kalshi_ws"

def load_bronze_channel(channel: str, date: str) -> list[dict]:
    merged_key = f"bronze_merged/{SOURCE}/{channel}/date={date}/merged.jsonl.gz"
    try:
        resp = s3.get_object(Bucket=BUCKET, Key=merged_key)
        data = gzip.decompress(resp["Body"].read()).decode()
        records = [json.loads(line) for line in data.split("\n") if line.strip()]
        print(f"  {channel}: {len(records):,} records (merged)")
        return records
    except s3.exceptions.NoSuchKey:
        # Try raw
        y, m, d = date.split("-")
        prefix = f"bronze/{SOURCE}/{channel}/{y}/{m}/{d}/"
        paginator = s3.get_paginator("list_objects_v2")
        keys = []
        for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
            keys.extend(o["Key"] for o in page.get("Contents", []))
        keys = [k for k in keys if k.endswith(".jsonl.gz")]
        records = []
        for k in keys:
            resp = s3.get_object(Bucket=BUCKET, Key=k)
            raw = gzip.decompress(resp["Body"].read()).decode()
            for line in raw.split("\n"):
                if line.strip():
                    records.append(json.loads(line))
        print(f"  {channel}: {len(records):,} records ({len(keys)} raw files)")
        return records

DATE = "2026-04-18"
print(f"Loading bronze for {DATE}...")
snap_records = load_bronze_channel("orderbook_snapshot", DATE)
delta_records = load_bronze_channel("orderbook_delta", DATE)
trade_records = load_bronze_channel("trade", DATE)
print(f"\nTotal: {len(snap_records):,} snapshots, {len(delta_records):,} deltas, {len(trade_records):,} trades")

## Step 1: Check if records have conn_id and sid

In [ ]:
# Check top-level fields in records
print("=== Sample snapshot record (top-level keys) ===")
if snap_records:
    print(json.dumps({k: type(v).__name__ for k, v in snap_records[0].items()}, indent=2))
    print("\nFull sample:")
    print(json.dumps(snap_records[0], indent=2, default=str)[:2000])

print("\n=== Sample delta record (top-level keys) ===")
if delta_records:
    print(json.dumps({k: type(v).__name__ for k, v in delta_records[0].items()}, indent=2))
    print("\nFull sample:")
    print(json.dumps(delta_records[0], indent=2, default=str)[:2000])

print("\n=== Sample trade record (top-level keys) ===")
if trade_records:
    print(json.dumps({k: type(v).__name__ for k, v in trade_records[0].items()}, indent=2))
else:
    print("  (no trade records)")

In [ ]:
# Check conn_id and sid presence across ALL records
all_records = snap_records + delta_records + trade_records

has_conn_id = sum(1 for r in all_records if "conn_id" in r)
has_frame = sum(1 for r in all_records if "frame" in r)
has_sid_in_frame = sum(1 for r in all_records if r.get("frame", {}).get("sid") is not None)
has_seq_in_frame = sum(1 for r in all_records if r.get("frame", {}).get("seq") is not None)

print(f"Total records: {len(all_records):,}")
print(f"  has conn_id: {has_conn_id:,} ({has_conn_id/len(all_records)*100:.1f}%)")
print(f"  has frame:   {has_frame:,} ({has_frame/len(all_records)*100:.1f}%)")
print(f"  has frame.sid: {has_sid_in_frame:,} ({has_sid_in_frame/len(all_records)*100:.1f}%)")
print(f"  has frame.seq: {has_seq_in_frame:,} ({has_seq_in_frame/len(all_records)*100:.1f}%)")

# Unique conn_ids
conn_ids = set(r.get("conn_id") for r in all_records if "conn_id" in r)
print(f"\nUnique conn_ids: {len(conn_ids)}")
for cid in sorted(conn_ids)[:10]:
    count = sum(1 for r in all_records if r.get("conn_id") == cid)
    print(f"  {cid}: {count:,} records")

# Unique sids in delta frames
delta_sids = Counter(r.get("frame", {}).get("sid") for r in delta_records)
snap_sids = Counter(r.get("frame", {}).get("sid") for r in snap_records)
print(f"\nDelta SIDs: {dict(delta_sids)}")
print(f"Snapshot SIDs: {dict(snap_sids)}")

## Step 2: Replay through KalshiTransform with diagnostics

Run the exact same transform that build_silver uses, but track when/why books cross.

In [ ]:
import sys
sys.path.insert(0, "/Users/chriswang/Desktop/prediction-market-exploration")

from v2.app.transforms.kalshi_ws import KalshiTransform
from v2.app.core.book_state import BookState

# Sort all records by t_receipt (same as build_silver)
all_sorted = sorted(all_records, key=lambda r: r.get("t_receipt", 0.0))
print(f"Replaying {len(all_sorted):,} records through KalshiTransform...")

tf = KalshiTransform()
total_depth = 0
crossed_count = 0
first_cross = {}  # ticker -> frame_num of first cross
cross_samples = []  # first N crossing events

for i, rec in enumerate(all_sorted):
    frame = rec.get("frame", {})
    t_receipt = rec.get("t_receipt", 0.0)
    conn_id = rec.get("conn_id")
    result = tf(frame, t_receipt, conn_id=conn_id)
    
    for row in result.depth_rows:
        total_depth += 1
        spread = row.get("spread")
        if spread is not None and spread < 0:
            crossed_count += 1
            ticker = row["market_ticker"]
            if ticker not in first_cross:
                first_cross[ticker] = i
            if len(cross_samples) < 20:
                cross_samples.append({
                    "frame_num": i,
                    "ticker": ticker,
                    "spread": spread,
                    "bid_1": row.get("bid_1"),
                    "ask_1": row.get("ask_1"),
                    "bid_1_size": row.get("bid_1_size"),
                    "ask_1_size": row.get("ask_1_size"),
                    "frame_type": frame.get("type"),
                    "frame_sid": frame.get("sid"),
                    "conn_id": conn_id,
                })

rate = crossed_count / max(total_depth, 1) * 100
print(f"\nResults:")
print(f"  Total depth rows: {total_depth:,}")
print(f"  Crossed: {crossed_count:,} ({rate:.1f}%)")
print(f"  Tickers with crosses: {len(first_cross)}")
print(f"  Books in transform: {len(tf._books)}")

print(f"\nFirst 20 crossing events:")
for s in cross_samples:
    print(f"  frame {s['frame_num']:>6}: {s['ticker']:<45} spread={s['spread']:>4} "
          f"bid={s['bid_1']} ask={s['ask_1']} type={s['frame_type']} sid={s['frame_sid']} conn={s['conn_id']}")

In [ ]:
# Which tickers cross and when?
crossed_tickers = sorted(first_cross.items(), key=lambda x: x[1])
print(f"Tickers that first cross (ordered by frame number):")
for ticker, frame_num in crossed_tickers[:30]:
    print(f"  frame {frame_num:>6}: {ticker}")

# What fraction of tickers cross?
all_tickers = set()
for rec in all_sorted:
    t = rec.get("frame", {}).get("msg", {}).get("market_ticker")
    if t:
        all_tickers.add(t)
print(f"\nTotal tickers: {len(all_tickers)}")
print(f"Tickers that cross: {len(first_cross)} ({len(first_cross)/len(all_tickers)*100:.0f}%)")

## Step 3: Deep dive on a specific crossing ticker

Pick the first ticker that crosses and trace exactly what happens to its book.

In [ ]:
# Pick the ticker that crosses earliest
TARGET = crossed_tickers[0][0] if crossed_tickers else None
print(f"Investigating: {TARGET}")

if TARGET:
    # Collect all records for this ticker
    target_recs = []
    for rec in all_sorted:
        msg = rec.get("frame", {}).get("msg", {})
        if msg.get("market_ticker") == TARGET:
            target_recs.append(rec)
    
    print(f"Records for {TARGET}: {len(target_recs):,}")
    
    # Count by type
    type_counts = Counter(r.get("frame", {}).get("type") for r in target_recs)
    print(f"By type: {dict(type_counts)}")
    
    # Check sids
    sids = Counter(r.get("frame", {}).get("sid") for r in target_recs)
    print(f"SIDs: {dict(sids)}")
    
    # Check conn_ids
    cids = Counter(r.get("conn_id") for r in target_recs)
    print(f"conn_ids: {dict(cids)}")
    
    # Time range
    times = [r.get("t_receipt", 0) for r in target_recs]
    print(f"Time range: {pd.Timestamp(min(times), unit='s', tz='UTC')} → {pd.Timestamp(max(times), unit='s', tz='UTC')}")
    print(f"Duration: {max(times) - min(times):.0f}s ({(max(times)-min(times))/3600:.1f} hours)")

In [ ]:
# Replay JUST this ticker manually, logging every step
if TARGET:
    from v2.app.core.conversions import dollars_to_cents
    
    book_yes = {}  # price -> size
    book_no = {}   # price -> size
    book_sid = None
    initialized = False
    
    log_entries = []  # (idx, type, detail, spread, crossed)
    
    for idx, rec in enumerate(target_recs):
        frame = rec.get("frame", {})
        msg = frame.get("msg", {})
        ftype = frame.get("type")
        sid = frame.get("sid")
        conn_id = rec.get("conn_id")
        
        if ftype == "orderbook_snapshot":
            book_yes = {}
            book_no = {}
            for ps, ss in msg.get("yes_dollars_fp", []):
                p = dollars_to_cents(ps)
                s = int(round(float(ss)))
                if s > 0:
                    book_yes[p] = s
            for ps, ss in msg.get("no_dollars_fp", []):
                p = dollars_to_cents(ps)
                s = int(round(float(ss)))
                if s > 0:
                    book_no[p] = s
            book_sid = sid
            initialized = True
            detail = f"yes_levels={len(book_yes)} no_levels={len(book_no)} sid={sid}"
            
        elif ftype == "orderbook_delta" and initialized:
            # Check sid filter
            if sid is not None and book_sid is not None and sid != book_sid:
                detail = f"SKIPPED (sid={sid} != book_sid={book_sid})"
            else:
                pc = dollars_to_cents(msg["price_dollars"])
                dv = int(round(float(msg["delta_fp"])))
                side = msg["side"]
                target_book = book_yes if side == "yes" else book_no
                old_sz = target_book.get(pc, 0)
                new_sz = old_sz + dv
                if new_sz <= 0:
                    target_book.pop(pc, None)
                else:
                    target_book[pc] = new_sz
                detail = f"{side} {pc}c delta={dv:+d} ({old_sz}→{max(new_sz,0)}) sid={sid}"
        else:
            detail = f"type={ftype} init={initialized}"
        
        # Compute spread
        best_bid = max(book_yes.keys()) if book_yes else None
        best_ask = (100 - max(book_no.keys())) if book_no else None
        spread = (best_ask - best_bid) if (best_bid is not None and best_ask is not None) else None
        crossed = spread is not None and spread < 0
        
        log_entries.append((idx, ftype, detail, spread, crossed, sid, conn_id))
    
    # Find first cross
    first_cross_idx = next((i for i, (_, _, _, _, c, _, _) in enumerate(log_entries) if c), None)
    
    if first_cross_idx is not None:
        print(f"First cross at record {first_cross_idx}")
        print(f"\nContext around first cross (5 before, 10 after):")
        start = max(0, first_cross_idx - 5)
        end = min(len(log_entries), first_cross_idx + 11)
        print(f"{'idx':>5} {'type':<22} {'sid':>4} {'spread':>7} {'X':>2} detail")
        print("-" * 100)
        for i in range(start, end):
            idx, ftype, detail, spread, crossed, sid, conn_id = log_entries[i]
            marker = "XX" if crossed else ""
            sp_str = f"{spread}" if spread is not None else "None"
            print(f"{idx:>5} {ftype or '?':<22} {sid!s:>4} {sp_str:>7} {marker:>2} {detail[:80]}")
    else:
        print("No crosses found in manual replay (unexpected!)")
        print("Checking if conn_id filter explains it...")

In [ ]:
# Check: is the issue that multiple SIDs are interleaved WITHOUT a new snapshot?
# i.e., sid changes from 1→2 in the delta stream but we never get a snapshot for sid=2

if TARGET:
    # Track sid transitions in the delta stream for this ticker
    delta_recs = [r for r in target_recs if r.get("frame", {}).get("type") == "orderbook_delta"]
    snap_recs = [r for r in target_recs if r.get("frame", {}).get("type") == "orderbook_snapshot"]
    
    print(f"Snapshots: {len(snap_recs)}")
    for i, r in enumerate(snap_recs):
        f = r.get("frame", {})
        print(f"  snap {i}: sid={f.get('sid')} seq={f.get('seq')} t={pd.Timestamp(r.get('t_receipt',0), unit='s', tz='UTC')}")
    
    print(f"\nDeltas: {len(delta_recs):,}")
    delta_sid_seq = [(r.get("frame",{}).get("sid"), r.get("frame",{}).get("seq"), r.get("t_receipt",0))
                     for r in delta_recs]
    
    # Find sid transitions
    transitions = []
    prev_sid = None
    for i, (sid, seq, t) in enumerate(delta_sid_seq):
        if prev_sid is not None and sid != prev_sid:
            transitions.append((i, prev_sid, sid, t))
        prev_sid = sid
    
    print(f"\nSID transitions in delta stream: {len(transitions)}")
    for i, (idx, old, new, t) in enumerate(transitions[:20]):
        print(f"  delta[{idx}]: sid {old} → {new} at {pd.Timestamp(t, unit='s', tz='UTC')}")
    
    # Key question: when sid changes, does a new snapshot arrive BEFORE or AFTER?
    snap_times = [r.get("t_receipt", 0) for r in snap_recs]
    snap_sids_list = [r.get("frame", {}).get("sid") for r in snap_recs]
    print(f"\nSnapshot SIDs: {snap_sids_list}")
    print(f"Delta SID distribution: {Counter(s for s, _, _ in delta_sid_seq)}")

## Step 4: Check a newer date (with conn_id) for comparison

Compare field presence on a newer date that passes validation.

In [ ]:
# Check a newer date for comparison
DATE_NEW = "2026-04-27"
print(f"Loading sample from {DATE_NEW}...")
delta_new = load_bronze_channel("orderbook_delta", DATE_NEW)

# Check fields
sample = delta_new[:5] if delta_new else []
print(f"\nSample record keys: {list(sample[0].keys()) if sample else 'N/A'}")
if sample:
    print(f"Sample record:")
    print(json.dumps(sample[0], indent=2, default=str)[:1000])

has_conn_new = sum(1 for r in delta_new[:1000] if "conn_id" in r)
has_sid_new = sum(1 for r in delta_new[:1000] if r.get("frame", {}).get("sid") is not None)
print(f"\nFirst 1000 delta records:")
print(f"  has conn_id: {has_conn_new}/1000")
print(f"  has frame.sid: {has_sid_new}/1000")

conn_ids_new = Counter(r.get("conn_id") for r in delta_new)
print(f"\nAll conn_ids: {dict(list(conn_ids_new.items())[:10])}")
sids_new = Counter(r.get("frame", {}).get("sid") for r in delta_new)
print(f"All delta SIDs: {dict(sids_new)}")

## Step 5: Understand the crossing mechanism

A crossed book means best_bid >= best_ask, i.e., max(yes) >= 100 - max(no).
This means max(yes) + max(no) >= 100.

For a YES-NO pair, this means the YES and NO books overlap in implied price space.
Possible causes:
1. Deltas applied from wrong subscription (stale sid) inflate one side
2. Missed snapshot after reconnect means book is stale
3. Deltas arriving out of order or duplicated

In [ ]:
# For the target ticker, show the book state at the point of first cross
# AND what the snapshot (if any) said

if TARGET and first_cross_idx is not None:
    # Replay again up to the first cross, capturing full book state
    book_yes2 = {}
    book_no2 = {}
    book_sid2 = None
    init2 = False
    skipped_deltas = 0
    applied_deltas = 0
    
    for idx, rec in enumerate(target_recs[:first_cross_idx + 1]):
        frame = rec.get("frame", {})
        msg = frame.get("msg", {})
        ftype = frame.get("type")
        sid = frame.get("sid")
        
        if ftype == "orderbook_snapshot":
            book_yes2 = {}
            book_no2 = {}
            for ps, ss in msg.get("yes_dollars_fp", []):
                p = dollars_to_cents(ps)
                s = int(round(float(ss)))
                if s > 0:
                    book_yes2[p] = s
            for ps, ss in msg.get("no_dollars_fp", []):
                p = dollars_to_cents(ps)
                s = int(round(float(ss)))
                if s > 0:
                    book_no2[p] = s
            book_sid2 = sid
            init2 = True
        elif ftype == "orderbook_delta" and init2:
            if sid is not None and book_sid2 is not None and sid != book_sid2:
                skipped_deltas += 1
            else:
                pc = dollars_to_cents(msg["price_dollars"])
                dv = int(round(float(msg["delta_fp"])))
                side = msg["side"]
                target_book = book_yes2 if side == "yes" else book_no2
                new_sz = target_book.get(pc, 0) + dv
                if new_sz <= 0:
                    target_book.pop(pc, None)
                else:
                    target_book[pc] = new_sz
                applied_deltas += 1
    
    best_bid = max(book_yes2.keys()) if book_yes2 else None
    best_ask = (100 - max(book_no2.keys())) if book_no2 else None
    
    print(f"Book state at first cross (record {first_cross_idx}):")
    print(f"  Applied deltas: {applied_deltas:,}, Skipped (sid mismatch): {skipped_deltas:,}")
    print(f"  Best bid (max yes): {best_bid}c")
    print(f"  Best ask (100 - max no): {best_ask}c")
    print(f"  Spread: {best_ask - best_bid if best_bid and best_ask else None}c")
    print(f"\n  YES book (top 10 levels):")
    for p, s in sorted(book_yes2.items(), reverse=True)[:10]:
        print(f"    {p}c: {s:,}")
    print(f"\n  NO book (top 10 levels):")
    for p, s in sorted(book_no2.items(), reverse=True)[:10]:
        implied_ask = 100 - p
        print(f"    {p}c (ask={implied_ask}c): {s:,}")
    
    # What was the last snapshot for this ticker?
    print(f"\n  Last snapshot had: yes_levels={len([r for r in snap_recs])} snapshots total")

## Step 6: Test fix — what if we skip dates without conn_id?

Or: what if we invalidate books when sid changes in delta stream (even without a new snapshot)?

In [ ]:
# Replay with MORE AGGRESSIVE sid handling:
# If a delta arrives with a DIFFERENT sid than book.sid, AND no snapshot
# has arrived for the new sid, RESET the book from this delta's sid
# (i.e., clear the book and start fresh, accepting deltas from new sid)

# Actually, let's first test: what if we just ALWAYS update book.sid
# to match the incoming delta's sid? This would mean after a reconnect,
# the first delta from the new sid would be applied and subsequent old-sid
# deltas would be rejected.

# But the real fix might be: detect when ALL deltas switch to a new sid
# (indicating a reconnect) and clear books that don't get a fresh snapshot.

# Let's test approach: "clear book on sid change in delta stream"
print("=== Replay with aggressive sid handling ===")
print("Policy: when delta.sid != book.sid, CLEAR the book (treat as stale)")
print("and skip the delta (wait for next snapshot).\n")

book_states = {}  # ticker -> {yes: {}, no: {}, sid: int}
total_depth_agg = 0
crossed_agg = 0
books_cleared = 0

for rec in all_sorted:
    frame = rec.get("frame", {})
    msg = frame.get("msg", {})
    ftype = frame.get("type")
    sid = frame.get("sid")
    ticker = msg.get("market_ticker")
    if not ticker:
        continue
    
    if ftype == "orderbook_snapshot":
        yes = {}
        no = {}
        for ps, ss in msg.get("yes_dollars_fp", []):
            p = dollars_to_cents(ps)
            s = int(round(float(ss)))
            if s > 0:
                yes[p] = s
        for ps, ss in msg.get("no_dollars_fp", []):
            p = dollars_to_cents(ps)
            s = int(round(float(ss)))
            if s > 0:
                no[p] = s
        book_states[ticker] = {"yes": yes, "no": no, "sid": sid}
        
        # Emit depth check
        best_bid = max(yes.keys()) if yes else None
        best_ask = (100 - max(no.keys())) if no else None
        if best_bid is not None and best_ask is not None:
            total_depth_agg += 1
            if best_ask - best_bid < 0:
                crossed_agg += 1
    
    elif ftype == "orderbook_delta":
        bs = book_states.get(ticker)
        if bs is None:
            continue
        
        # Aggressive sid check: clear on mismatch
        if sid is not None and bs["sid"] is not None and sid != bs["sid"]:
            # Book is stale — clear it and wait for snapshot
            del book_states[ticker]
            books_cleared += 1
            continue
        
        pc = dollars_to_cents(msg["price_dollars"])
        dv = int(round(float(msg["delta_fp"])))
        side = msg["side"]
        target_book = bs["yes"] if side == "yes" else bs["no"]
        new_sz = target_book.get(pc, 0) + dv
        if new_sz <= 0:
            target_book.pop(pc, None)
        else:
            target_book[pc] = new_sz
        
        # Emit depth check
        best_bid = max(bs["yes"].keys()) if bs["yes"] else None
        best_ask = (100 - max(bs["no"].keys())) if bs["no"] else None
        if best_bid is not None and best_ask is not None:
            total_depth_agg += 1
            if best_ask - best_bid < 0:
                crossed_agg += 1

rate_agg = crossed_agg / max(total_depth_agg, 1) * 100
print(f"Results with aggressive sid handling:")
print(f"  Total depth rows: {total_depth_agg:,}")
print(f"  Crossed: {crossed_agg:,} ({rate_agg:.1f}%)")
print(f"  Books cleared on sid mismatch: {books_cleared:,}")
print(f"  Remaining active books: {len(book_states)}")

In [ ]:
# Alternative approach: what if the issue is that conn_id is missing?
# Without conn_id, the transform can't detect reconnects.
# Let's detect reconnects by looking for GAPS in the seq numbers.

# For each sid, find the seq range
print("=== Sequence analysis for delta records ===")

# Group by sid
seqs_by_sid = defaultdict(list)
for rec in delta_records:
    sid = rec.get("frame", {}).get("sid")
    seq = rec.get("frame", {}).get("seq")
    if sid is not None and seq is not None:
        seqs_by_sid[sid].append(seq)

for sid in sorted(seqs_by_sid.keys()):
    seqs = sorted(seqs_by_sid[sid])
    gaps = [(seqs[i+1] - seqs[i]) for i in range(len(seqs)-1) if seqs[i+1] - seqs[i] > 1]
    print(f"  SID {sid}: {len(seqs):,} records, seq {seqs[0]}→{seqs[-1]}, "
          f"gaps>1: {len(gaps)} (max gap: {max(gaps) if gaps else 0})")

# Same for snapshots
print("\n=== Sequence analysis for snapshot records ===")
snap_seqs_by_sid = defaultdict(list)
for rec in snap_records:
    sid = rec.get("frame", {}).get("sid")
    seq = rec.get("frame", {}).get("seq")
    if sid is not None and seq is not None:
        snap_seqs_by_sid[sid].append(seq)

for sid in sorted(snap_seqs_by_sid.keys()):
    seqs = sorted(snap_seqs_by_sid[sid])
    print(f"  SID {sid}: {len(seqs)} snapshots, seq range {seqs[0]}→{seqs[-1]}")

In [ ]:
# Key hypothesis: if snapshots are on sid=X but deltas switch from sid=X to sid=Y
# (due to reconnect), and no new snapshot arrives on sid=Y, then:
# - The current code (sid filter) CORRECTLY skips sid=Y deltas
# - BUT the book is now frozen at its last state from sid=X snapshots/deltas
# - If the market has moved, the frozen book will show stale prices = CROSSED
#
# The fix: when we detect that we're only receiving deltas from a NEW sid
# (and no matching snapshot), we should INVALIDATE the book instead of
# keeping it frozen.
#
# Alternative: the problem is that sid is None for some/all records on this date.
# Let's check:

print("=== Checking None sids ===")
delta_none_sid = sum(1 for r in delta_records if r.get("frame", {}).get("sid") is None)
snap_none_sid = sum(1 for r in snap_records if r.get("frame", {}).get("sid") is None)
print(f"Deltas with sid=None: {delta_none_sid:,} / {len(delta_records):,}")
print(f"Snapshots with sid=None: {snap_none_sid:,} / {len(snap_records):,}")

# If sid is None, the filter `if sid is not None and book.sid is not None` is False
# So ALL deltas would be applied regardless — no protection!
if delta_none_sid > 0:
    print(f"\n>>> {delta_none_sid:,} deltas have sid=None — sid filter is BYPASSED!")
    print("    ALL deltas are applied regardless of subscription state.")
    print("    After a reconnect, old AND new deltas both get applied → drift → crosses.")
else:
    print("\n>>> All deltas have sid set — filter should work.")
    print("    Issue might be that book stays frozen when deltas switch to new sid.")

## Step 7: Test the actual fix

Based on findings, test the correct fix:
- If sid is None: we need conn_id-based detection (or skip this date)
- If sid is set but book freezes: invalidate books that stop receiving matching deltas

In [ ]:
# DEFINITIVE TEST: Replay through KalshiTransform but with a fix that
# invalidates (removes) books when we detect they're stale.
#
# Staleness detection: if we see N consecutive deltas for a ticker with
# a DIFFERENT sid than the book, the book is stale.
#
# Alternative if sid is None: use conn_id transitions, or just
# detect that we haven't had a snapshot in a while.

from v2.app.transforms.kalshi_ws import KalshiTransform

# Modified replay: track consecutive skips per ticker
tf2 = KalshiTransform()
total_depth2 = 0
crossed2 = 0
invalidated2 = 0
skip_counts = defaultdict(int)  # ticker -> consecutive skips
SKIP_THRESHOLD = 5  # after N skipped deltas, invalidate

for rec in all_sorted:
    frame = rec.get("frame", {})
    t_receipt = rec.get("t_receipt", 0.0)
    conn_id = rec.get("conn_id")
    msg = frame.get("msg", {})
    ticker = msg.get("market_ticker", "")
    ftype = frame.get("type")
    sid = frame.get("sid")
    
    # Pre-check: would this delta be skipped by sid filter?
    if ftype == "orderbook_delta" and ticker in tf2._books:
        book = tf2._books[ticker]
        if sid is not None and book.sid is not None and sid != book.sid:
            skip_counts[ticker] += 1
            if skip_counts[ticker] >= SKIP_THRESHOLD:
                # This book is stale — remove it
                del tf2._books[ticker]
                invalidated2 += 1
                skip_counts[ticker] = 0
    elif ftype == "orderbook_delta" and ticker in tf2._books:
        skip_counts[ticker] = 0  # reset on successful application
    
    # Normal transform
    result = tf2(frame, t_receipt, conn_id=conn_id)
    
    for row in result.depth_rows:
        total_depth2 += 1
        spread = row.get("spread")
        if spread is not None and spread < 0:
            crossed2 += 1

rate2 = crossed2 / max(total_depth2, 1) * 100
print(f"=== With stale-book invalidation (threshold={SKIP_THRESHOLD}) ===")
print(f"  Total depth rows: {total_depth2:,}")
print(f"  Crossed: {crossed2:,} ({rate2:.1f}%)")
print(f"  Books invalidated: {invalidated2:,}")
print(f"  Active books: {len(tf2._books)}")
print(f"\n  Compare: original was 41.1% crossed")

In [ ]:
# Also test: what happens if we just DON'T emit depth rows for books
# where the last applied delta was > N seconds ago?
# This is a simpler heuristic: "stale book = don't report it"

# But the real question is: what's the root cause?
# Let's figure out if it's:
# A) sid=None (no filtering possible)
# B) sid filter works but book stays frozen and crosses
# C) something else entirely

# Count how many depth rows come from snapshots vs deltas
tf3 = KalshiTransform()
depth_from_snap = 0
depth_from_delta = 0
crossed_from_snap = 0
crossed_from_delta = 0

for rec in all_sorted:
    frame = rec.get("frame", {})
    t_receipt = rec.get("t_receipt", 0.0)
    conn_id = rec.get("conn_id")
    ftype = frame.get("type")
    result = tf3(frame, t_receipt, conn_id=conn_id)
    
    for row in result.depth_rows:
        spread = row.get("spread")
        crossed = spread is not None and spread < 0
        if ftype == "orderbook_snapshot":
            depth_from_snap += 1
            if crossed:
                crossed_from_snap += 1
        elif ftype == "orderbook_delta":
            depth_from_delta += 1
            if crossed:
                crossed_from_delta += 1

print("=== Crossed by source ===")
print(f"From snapshots: {crossed_from_snap}/{depth_from_snap} "
      f"({crossed_from_snap/max(depth_from_snap,1)*100:.1f}%)")
print(f"From deltas:    {crossed_from_delta}/{depth_from_delta} "
      f"({crossed_from_delta/max(depth_from_delta,1)*100:.1f}%)")
print(f"\nIf snapshots are crossed, the exchange itself has crossed books (unlikely).")
print(f"If only deltas cross, it's drift from our reconstruction.")

In [ ]:
# Final summary and proposed fix
print("="*70)
print("DIAGNOSIS SUMMARY")
print("="*70)
print(f"""
Date: {DATE}
Records: {len(snap_records)} snapshots, {len(delta_records):,} deltas, {len(trade_records)} trades
Crossed rate: {rate:.1f}% (threshold: 5%)

Key findings:
- conn_id present: {has_conn_id}/{len(all_records)} records
- frame.sid present: {has_sid_in_frame}/{len(all_records)} records
- Crossed from snapshots: {crossed_from_snap} (exchange-side crosses)
- Crossed from deltas: {crossed_from_delta} (reconstruction drift)

Root cause analysis:
""")

if has_conn_id == 0 and has_sid_in_frame == 0:
    print("  >>> NEITHER conn_id nor sid present in bronze data.")
    print("  >>> ALL protection mechanisms are bypassed.")
    print("  >>> Fix: skip dates without conn_id/sid, OR require merged bronze")
    print("      with conn_id added by merge_bronze.")
elif has_conn_id == 0 and has_sid_in_frame > 0:
    print("  >>> No conn_id but sid IS present.")
    print("  >>> The sid filter works, but frozen books (book.sid != delta sid)")
    print("      keep emitting stale depth rows that eventually show as crossed.")
    print("  >>> Fix: invalidate books after N consecutive sid-mismatch skips.")
elif has_conn_id > 0:
    print("  >>> conn_id IS present — reconnect detection should work.")
    print("  >>> Need to investigate further why books still cross.")
    print("  >>> Possible: conn_id only partially populated.")